# Marketing Attribution — LTV & Payback Period
## Business Questions
- What is the lifetime value (LTV) of a customer from each channel?
- How long does it take to recover CAC (Payback Period)?
- Which channel has the best LTV/CAC ratio?

**Formula:** `LTV = ARPU × Gross Margin / Monthly Churn`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", font_scale=1.1)

ROOT = Path("..")
IMG  = ROOT / "images"

spend = pd.read_csv(ROOT / "data" / "marketing_spend.csv")
acq   = pd.read_csv(ROOT / "data" / "customer_acquisitions.csv")
ltv   = pd.read_csv(ROOT / "data" / "cohort_ltv.csv")

COLORS = {
    "google_ads":   "#2a9d8f",
    "facebook_ads": "#e76f51",
    "seo":          "#264653",
    "email":        "#e9c46a",
}
CHANNEL_LABELS = {
    "google_ads": "Google Ads",
    "facebook_ads": "Facebook Ads",
    "seo": "SEO",
    "email": "Email",
}
print("Data loaded ✓")
print(f"  Spend rows   : {len(spend):,}")
print(f"  Acq rows     : {len(acq):,}")
print(f"  LTV rows     : {len(ltv):,}")

## 1 · Survival Curve — Customer Retention by Channel

In [ ]:
# Average retention over cohort months 0-12
survival = (
    ltv.groupby(["channel","cohort_month"])
    .agg(avg_customers=("customers_active","mean"))
    .reset_index()
)
# Normalise to month-0
m0 = survival[survival["cohort_month"]==0].set_index("channel")["avg_customers"]
survival["retention_pct"] = survival.apply(
    lambda r: r["avg_customers"] / m0[r["channel"]], axis=1
)

fig, ax = plt.subplots(figsize=(12, 5))
for ch in survival["channel"].unique():
    d = survival[survival["channel"]==ch]
    ax.plot(d["cohort_month"], d["retention_pct"]*100,
            label=CHANNEL_LABELS[ch], color=COLORS[ch], linewidth=2, marker="o", markersize=4)

ax.set_xlabel("Months Since Acquisition")
ax.set_ylabel("Customers Retained (%)")
ax.set_title("Survival Curve by Channel")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f"{v:.0f}%"))
ax.legend()
plt.tight_layout()
plt.savefig(IMG / "05_survival_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 2 · LTV Calculation

In [ ]:
CHANNEL_PARAMS = {
    "google_ads":   {"arpu": 48.0, "margin": 0.72, "churn": 0.055, "cac": 78.8},
    "facebook_ads": {"arpu": 39.0, "margin": 0.70, "churn": 0.082, "cac": 88.9},
    "seo":          {"arpu": 55.0, "margin": 0.76, "churn": 0.038, "cac": 14.9},
    "email":        {"arpu": 43.0, "margin": 0.74, "churn": 0.028, "cac": 10.7},
}

results = []
for ch, p in CHANNEL_PARAMS.items():
    ltv_v        = p["arpu"] * p["margin"] / p["churn"]
    payback      = p["cac"] / (p["arpu"] * p["margin"])
    ltv_cac      = ltv_v / p["cac"]
    results.append({
        "Channel":        CHANNEL_LABELS[ch],
        "ARPU ($/mo)":    p["arpu"],
        "Gross Margin":   f"{p['margin']:.0%}",
        "Monthly Churn":  f"{p['churn']:.1%}",
        "CAC ($)":        p["cac"],
        "LTV ($)":        round(ltv_v, 0),
        "LTV/CAC":        round(ltv_cac, 1),
        "Payback (mo)":   round(payback, 1),
    })

df_res = pd.DataFrame(results).set_index("Channel")
print(df_res.to_string())

In [ ]:
# LTV/CAC bar chart with threshold lines
ltv_cac = {CHANNEL_LABELS[ch]: p["arpu"]*p["margin"]/p["churn"]/p["cac"]
           for ch, p in CHANNEL_PARAMS.items()}
ltv_cac = dict(sorted(ltv_cac.items(), key=lambda x: x[1]))

fig, ax = plt.subplots(figsize=(9, 5))
colors = [COLORS[k] for k in ["email","seo","google_ads","facebook_ads"]]
bars = ax.barh(list(ltv_cac.keys()), list(ltv_cac.values()),
               color=colors, edgecolor="white")
ax.axvline(3,  color="#e76f51", linestyle="--", label="Minimum: 3x")
ax.axvline(10, color="#2a9d8f", linestyle="--", label="Good: 10x")
ax.set_xlabel("LTV / CAC Ratio")
ax.set_title("LTV/CAC Ratio by Channel")
ax.legend()
for bar, v in zip(bars, ltv_cac.values()):
    ax.text(v + 0.5, bar.get_y()+bar.get_height()/2, f"{v:.1f}x",
            va="center", fontweight="bold")
plt.tight_layout()
plt.savefig(IMG / "06_ltv_cac_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

## 3 · Payback Period — Cumulative Gross Profit vs CAC

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
months = np.arange(0, 25)

for ch, p in CHANNEL_PARAMS.items():
    cum_gp = np.array([
        p["arpu"] * p["margin"] * (1-(1-p["churn"])**t) / p["churn"]
        for t in months
    ])
    ax.plot(months, cum_gp, label=CHANNEL_LABELS[ch],
            color=COLORS[ch], linewidth=2)
    # Mark payback point
    pb = p["cac"] / (p["arpu"] * p["margin"])
    ax.axhline(p["cac"], color=COLORS[ch], linestyle=":", alpha=0.4)

ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("Months Since Acquisition")
ax.set_ylabel("Cumulative Gross Profit per Customer ($)")
ax.set_title("Payback Period — When Does Gross Profit Recover CAC?")
ax.legend()
plt.tight_layout()
plt.savefig(IMG / "07_payback_period.png", dpi=150, bbox_inches="tight")
plt.show()

for ch, p in CHANNEL_PARAMS.items():
    pb = p["cac"] / (p["arpu"] * p["margin"])
    print(f"  {CHANNEL_LABELS[ch]:<14}: Payback in {pb:.1f} months  (CAC=${p['cac']:.0f})")